In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
pip install transformers datasets peft accelerate evaluate jiwer librosa audiomentations mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 120.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 103.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s  0:00:00
  Attempting uninstall: soxr━━━━━━━━━━━━━━━━━━━━  0/21 [huey]
    Found existing installation: soxr 1.1.0━  0/21 [huey]
    Uninstalling soxr-1.1.0:━━━━━━━━━━━━━━━━  0/21 [huey]
      Successfully uninstalled soxr-1.1.0━━━  0/21 [huey]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [mlflow]20/21 [mlflow]e]inny]]roto]
Note: you may need to restart the kernel to use updated packages.


In [6]:
from datasets import load_dataset, Audio
ds = load_dataset("google/fleurs", "hi_in")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/hi_in/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.53GB            

parquet-data/hi_in/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/hi_in/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  162MB            

parquet-data/hi_in/validation-00000-of-0(…): downloading bytes:           |  0.00B            

parquet-data/hi_in/test-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B /  306MB            

parquet-data/hi_in/test-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/239 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/418 [00:00<?, ? examples/s]

In [8]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [9]:
import re, unicodedata
def normalize(t):
    t = "".join(" " if unicodedata.category(c).startswith("P") else c for c in t)
    return re.sub(r"\s+", " ", t).strip().lower()

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small", torch_dtype=torch.float16).to("cuda").eval()

In [ ]:
test = ds["test"]; preds = []
for i in range(0, len(test), 16):
    batch = test[i:i+16]
    feats = processor([a["array"] for a in batch["audio"]], sampling_rate=16000,
                      return_tensors="pt").input_features.to("cuda", torch.float16)
    with torch.no_grad():
        ids = model.generate(feats, language="hindi", task="transcribe", max_new_tokens=225)
    preds += processor.batch_decode(ids, skip_special_tokens=True)

In [ ]:
import jiwer, pandas as pd
refs = [normalize(t) for t in test["transcription"]]
hyps = [normalize(p) for p in preds]
print("WER:", jiwer.wer(refs, hyps), "CER:", jiwer.cer(refs, hyps))
pd.DataFrame({"ref": refs, "hyp": hyps}).to_csv("/kaggle/working/baseline_preds.csv", index=False)

In [ ]:
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="hindi", task="transcribe")
def prep(b):
    b["input_features"] = processor.feature_extractor(b["audio"]["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["transcription"]).input_ids
    return b
short = lambda n: n < 30 * 16000
train = ds["train"].filter(short, input_columns="num_samples").map(prep, remove_columns=ds["train"].column_names)
val = ds["validation"].filter(short, input_columns="num_samples").map(prep, remove_columns=ds["validation"].column_names)

In [ ]:
import peft.tuners.lora.torchao as lora_torchao
lora_torchao.is_torchao_available = lambda: False

In [ ]:
import peft.tuners.lora.torchao as lora_torchao
lora_torchao.is_torchao_available = lambda: False

class Collator:
    def __init__(self, p): self.p = p
    def __call__(self, feats):
        x = self.p.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        y = self.p.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = y["input_ids"].masked_fill(y.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.p.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")).all():
            labels = labels[:, 1:]
        x["labels"] = labels
        return x

from peft import LoraConfig, get_peft_model
if "model" in globals():
    del model
torch.cuda.empty_cache()
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.generation_config.language, model.generation_config.task = "hindi", "transcribe"
model.generation_config.forced_decoder_ids = None
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none"))
model.print_trainable_parameters()

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper-small-hi-lora",
    per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=1e-3, warmup_steps=50, max_steps=600, fp16=True,
    eval_strategy="steps", eval_steps=150, save_steps=150, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=25, dataloader_num_workers=2,
    remove_unused_columns=False, label_names=["labels"], report_to="none")
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train, eval_dataset=val, data_collator=Collator(processor))
trainer.train()
model.save_pretrained("/kaggle/working/whisper-small-hi-lora-best")
model.push_to_hub("whisper-small-hi-lora")

In [10]:
import torch
import peft.tuners.lora.torchao as lora_torchao
lora_torchao.is_torchao_available = lambda: False
from peft import PeftModel
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
ft = PeftModel.from_pretrained(base, "Adi-13-adi/whisper-small-hi-lora")
ft = ft.merge_and_unload().half().to("cuda").eval()
from tqdm.auto import tqdm
test = ds["test"]
def transcribe(m):
    out = []
    for i in tqdm(range(0, len(test), 16)):
        b = test[i:i+16]
        f = processor([a["array"] for a in b["audio"]], sampling_rate=16000, return_tensors="pt").input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = m.generate(f, language="hindi", task="transcribe", max_new_tokens=440)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out
import jiwer, pandas as pd
ft_preds = transcribe(ft)
refs = [normalize(t) for t in test["transcription"]]
hyps = [normalize(p) for p in ft_preds]
print("WER:", jiwer.wer(refs, hyps), "CER:", jiwer.cer(refs, hyps))
pd.DataFrame({"ref": refs, "hyp": hyps}).to_csv("/kaggle/working/finetuned_preds.csv", index=False)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (9 outputs left out)

adapter_model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/27 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers a

WER: 0.2776486538834011 CER: 0.10397898628271232


In [9]:
import numpy as np
from audiomentations import AddGaussianSNR

def transcribe(m, snr=None):
    aug = AddGaussianSNR(min_snr_db=snr, max_snr_db=snr, p=1.0) if snr is not None else None
    out = []
    for i in tqdm(range(0, len(test), 16)):
        arrs = [a["array"].astype(np.float32) for a in test[i:i+16]["audio"]]
        if aug: arrs = [aug(x, sample_rate=16000) for x in arrs]
        f = processor(arrs, sampling_rate=16000, return_tensors="pt").input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = m.generate(f, language="hindi", task="transcribe", max_new_tokens=225)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out

base_m = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", torch_dtype=torch.float16).to("cuda").eval()
rows = []
for snr in [10, 5, 0]:
    for name, m in [("zero-shot", base_m), ("lora", ft)]:
        np.random.seed(0)
        h = [normalize(p) for p in transcribe(m, snr)]
        rows.append({"model": name, "snr_db": snr, "wer": jiwer.wer(refs, h), "cer": jiwer.cer(refs, h)})

noise_df = pd.DataFrame(rows)
noise_df.to_csv("/kaggle/working/noise_benchmark.csv", index=False)
noise_df

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  0%|          | 0/27 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  0%|          | 0/27 [00:00<?, ?it/s]

... (4 outputs left out)

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  0%|          | 0/27 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,model,snr_db,wer,cer
0,zero-shot,10,0.851249,0.508979
1,lora,10,0.484311,0.223173
2,zero-shot,5,0.931629,0.602257
3,lora,5,0.613403,0.321704
4,zero-shot,0,1.032055,0.748886
5,lora,0,0.750726,0.439031


In [3]:
!pip install -q jiwer
import jiwer, pandas as pd
import glob
path = glob.glob("/kaggle/input/**/finetuned_preds.csv", recursive=True)[0]
df = pd.read_csv(path).fillna("")
out = jiwer.process_words(df["ref"].tolist(), df["hyp"].tolist())
print(f"WER {out.wer:.3f} | substitutions {out.substitutions}, deletions {out.deletions}, insertions {out.insertions}, correct {out.hits}")
import re
from collections import Counter

def loose(s):  # treat nukta and chandrabindu/anusvara variants as equal
    return s.replace("\u093C", "").replace("\u0901", "\u0902")

def tag(r, h):
    if re.search(r"[0-9०-९]", r + h): return "numerals"
    if re.search(r"[^\u0900-\u097F]", h): return "non-Devanagari output"
    if loose(r) == loose(h): return "orthographic variant"
    if jiwer.cer(r, h) <= 0.34: return "near-miss spelling"
    return "wrong word"

pairs, tags = [], Counter()
for i, chunks in enumerate(out.alignments):
    R, H = out.references[i], out.hypotheses[i]
    for c in chunks:
        r, h = R[c.ref_start_idx:c.ref_end_idx], H[c.hyp_start_idx:c.hyp_end_idx]
        if c.type == "substitute":
            for rw, hw in zip(r, h):
                t = tag(rw, hw); tags[t] += 1; pairs.append((rw, hw, t))
        elif c.type == "delete": tags["deleted word"] += len(r)
        elif c.type == "insert": tags["inserted word"] += len(h)

summary = pd.Series(tags).sort_values(ascending=False)
print((summary / summary.sum() * 100).round(1).to_string())
print("\nStrict WER:", round(out.wer, 3), "| Lenient WER:",
      round(jiwer.wer([loose(x) for x in df.ref], [loose(x) for x in df.hyp]), 3))
for (r, h), n in Counter((r, h) for r, h, _ in pairs).most_common(20):
    print(f"{n:3d}  {r}  →  {h}")

df["wer"] = [jiwer.wer(r, h) for r, h in zip(df.ref, df.hyp)]
df.sort_values("wer", ascending=False).head(50).assign(manual_tag="") \
  .to_csv("/kaggle/working/worst50_for_review.csv", index=False)

WER 0.286 | substitutions 2402, deletions 334, insertions 213, correct 7590
wrong word               38.7
near-miss spelling       37.9
deleted word             11.3
inserted word             7.2
orthographic variant      2.7
numerals                  1.9
non-Devanagari output     0.3

Strict WER: 0.286 | Lenient WER: 0.278
 33  है  →  हैं
 22  हैं  →  है
 18  के  →  की
  8  कि  →  की
  7  पदार्थ  →  पदार्त
  5  ये  →  यह
  5  दक्षिण  →  दक्षिन
  4  सामान  →  समान
  4  सलाह  →  सला
  4  बोडा  →  बोड़ा
  4  की  →  के
  4  पार्क  →  पार
  4  है  →  हे
  4  वेबसाइट  →  वेपसाइट
  3  हानि  →  हानी
  3  या  →  यह
  3  की  →  कि
  3  यह  →  या
  3  सामाजिक  →  समाजिक
  3  मछली  →  मचली


In [ ]:
base_m = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", torch_dtype=torch.float16).to("cuda").eval()
hb = [normalize(p) for p in transcribe(base_m)]
print("WER:", jiwer.wer(refs, hb), "CER:", jiwer.cer(refs, hb))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface